# 012 — Proyecto: mapa evolutivo verificable de la IA

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

El proyecto integra la parte 00: una línea de tiempo de la IA donde **cada hito es una
afirmación falsable con fuente primaria**, clasificada en una taxonomía explícita y con su
limitación honesta. Protocolo por hito:

```text
1. Afirmación falsable (fecha + actor + resultado medible)
2. Fuente primaria (DOI / informe / propuesta original)
3. Contraste con 1 fuente secundaria (AIMA §1.1, Nilsson)
4. Limitación honesta (qué NO demuestra el hito)
5. Taxonomía en 3 ejes: paradigma (simbólico/conexionista/probabilístico/híbrido),
   tipo (teórico/algorítmico/sistema/datos-hardware/institucional),
   estado de la evidencia (primaria/secundaria/claim fallido)
6. Registro estructurado (JSON) que el laboratorio capstone valida
```

Regla clave: los **claims fallidos** (Simon 1965, Quinta Generación) se registran con la
misma disciplina que los éxitos — sin ellos, el mapa sufre sesgo de supervivencia y los
inviernos resultan inexplicables. El validador automático comprueba la *estructura*; la
calidad de las fuentes la evalúa la rúbrica humana: distinguir ambos niveles de validación
es en sí una lección del programa.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Afirmación: "En 1943, McCulloch y Pitts publican un modelo formal de
neurona binaria con umbral y demuestran que redes de estas unidades computan cualquier
función booleana". Fuente primaria: DOI 10.1007/BF02478259. Secundaria: AIMA §1.2 /
Nilsson cap. 2. Limitación: sin regla de aprendizaje (los pesos no se ajustan); no es un
modelo biológicamente fiel. Taxonomía: conexionista · teórico · evidencia primaria.

**Ejercicio 2.** El claim original: hacia el año 2000, un interrogador *promedio* no
tendría más de 70 % de probabilidad de identificar correctamente a la máquina tras
**5 minutos** de preguntas. La versión popular elimina el porcentaje, el tiempo y el tipo
de interrogador: sin esos parámetros, cualquier resultado puede declararse "cumplimiento" o
"fracaso" — es menos falsable precisamente porque es menos específica.

**Ejercicio 3.** El campo crítico es `claim`: hay que citar la predicción con su alcance y
plazo reales ("máquinas capaces de hacer cualquier trabajo que hace un hombre" en ~20
años, 1965) sin caricaturizarla ni suavizarla, porque el valor del hito fallido depende de
que la refutación sea justa. La limitación registra qué refuta exactamente el paso del
plazo (la predicción temporal) y qué no (la posibilidad en principio).

**Ejercicio 4.** Un validador de estructura acepta el hito con fecha 1946 si los campos son
correctos: la falsedad histórica es invisible para él. Reglas adicionales posibles:
consistencia cronológica entre hitos dependientes (Dartmouth > 1955 por la fecha de su
propuesta), listas cerradas de valores para la taxonomía, y verificación de que el DOI
resuelve. Aun así, la capa final (¿la fuente dice lo que el claim afirma?) es
irreduciblemente humana — la moraleja de la clase 010.

In [ ]:
result = run_lab("capstone", seed=12)
assert result["kind"] == "capstone"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — registro completo del hito 1943
hito_1943 = {
    "date": "1943",
    "claim": ("McCulloch y Pitts publican la neurona formal binaria con umbral y "
              "demuestran que sus redes computan cualquier función booleana"),
    "source": "https://doi.org/10.1007/BF02478259",
    "secondary": "AIMA 4e §1.2; Nilsson (2010) cap. 2",
    "taxonomy": ["conexionista", "teorico", "evidencia_primaria"],
    "limitations": ["sin regla de aprendizaje", "no es modelo biologico fiel"],
}
import json as _json
print(_json.dumps(hito_1943, ensure_ascii=False, indent=2))

In [ ]:
# Ejercicio 4 — validación estructural vs verdad histórica
def valida_estructura(hito):
    requeridos = {"date", "claim", "source", "taxonomy", "limitations"}
    return requeridos <= set(hito) and bool(hito["claim"]) and bool(hito["limitations"])

hito_falso = {
    "date": "1946",  # históricamente falso: la propuesta es de 1955, el taller de 1956
    "claim": "Se celebra el taller de Dartmouth",
    "source": "http://jmc.stanford.edu/articles/dartmouth/dartmouth.pdf",
    "taxonomy": ["institucional"],
    "limitations": ["ninguna"],
}
print("estructura válida:", valida_estructura(hito_falso))  # True: el validador no ve la mentira
# Regla adicional: rangos cronológicos conocidos por hito ancla
assert valida_estructura(hito_falso)  # demuestra el límite de la validación mecánica

## Reflexión

1. El laboratorio `capstone` valida estructura, no verdad. Da un ejemplo concreto de hito
   con JSON perfectamente válido y contenido históricamente falso. ¿Qué capa del protocolo
   lo detectaría?
2. ¿Por qué registrar la promesa de Simon (1965) como "claim fallido" mejora el mapa en
   lugar de ensuciarlo? Conéctalo con el sesgo de supervivencia de la clase 008.
3. De los 12 hitos ancla del esqueleto, ¿cuál te costó más verificar con fuente primaria y
   qué aprendiste sobre la diferencia entre "lo que todo el mundo repite" y lo que el
   documento original dice?